In [25]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity
import datetime as dt

df = pd.read_csv("C:/Users/deana/OneDrive/Documents/Eduvos/Degree/YEAR 3/ITDPA3/Fixed.csv", sep=";", dtype=str)

cosine_df["Date"] = pd.to_datetime(cosine_df["Date"], errors="coerce")

cosine_df = df.copy()

cosine_df = cosine_df.loc[(cosine_df["Date"] >= "2021-01-01") & (cosine_df["Date"] < "2025-06-01")]

#Cleaning job titles. Making sure the job title search is not case sensitive
cosine_df["Job_Title_Display"] = (cosine_df["Job_Title"].astype(str).str.strip())
cosine_df["Job_Title_Search"] = (cosine_df["Job_Title"].str.casefold())

#create a posting id for reference
cosine_df["Posting_ID"] = range(len(cosine_df))

#cleaning skills 
cosine_df["Skills"] = (cosine_df["Skills"].astype(str).str.replace(r"[\[\]\"']", "", regex=True))

#split skills into individual rows, making sure they're not case sensitive
skills_df = cosine_df.assign(Skills=cosine_df["Skills"].str.split(",")).explode("Skills")
skills_df["Skills"] = (skills_df["Skills"].str.strip().str.casefold())

#skills_df["Job_Title_Search"] = (skills_df["Job_Title_Search"].str.strip())

#skills_df["Skills"] = (skills_df["Skills"].str.strip().str.lower())

#removing duplicates skills from the datasets
skills_df = skills_df.drop_duplicates(subset=["Posting_ID", "Job_Title_Search", "Skills"])

#map titles to their job names
title_lookup = (skills_df[["Job_Title_Search", "Job_Title_Display"]].drop_duplicates(
    subset=["Job_Title_Search"]).set_index("Job_Title_Search")["Job_Title_Display"])

#creating a maxtrix job title x skill
matrix = pd.crosstab(skills_df["Job_Title_Search"], skills_df["Skills"])

#TF-IDF calculation and cosine similarity
tfidf = TfidfTransformer()
tfidf_matrix = tfidf.fit_transform(matrix)

similarity_matrix = cosine_similarity(tfidf_matrix)

job_title = input("Enter job title:").strip().casefold()

if job_title not in matrix.index: 
    print("Job Title not found")
else:
    #get similarity score for job
    title_index = matrix.index.get_loc(job_title)

    score = similarity_matrix[title_index]

    result = pd.DataFrame({ "Job_Title" : [title_lookup.get(title, title) for title in matrix.index],
                           "Cosine_Similarity" : score})
    
    similar_jobs = result[result.index != title_index].sort_values("Cosine_Similarity", ascending=False)

    print("\nTop 15 similar jobs:")
    print(result.sort_values("Cosine_Similarity", ascending=False).head(20).to_string(index=False, formatters={"Cosine_Similarity" : "{:.4f}".format}))

if not similar_jobs.empty:
    matches = similar_jobs.iloc[0]["Job_Title"]
    top_score = similar_jobs.iloc[0]["Cosine_Similarity"]

    print("\nMost similar job:")
    print(f"{matches} (Similarity: {top_score:.4f})")

    #show the skills for the entered job and the matched top job according to similarity score
    selected_skills = (matrix.loc[job_title].loc[lambda x: x>0].sort_values(ascending=False))
    
    top_job = matrix.index[result.index.get_loc(similar_jobs.index[0])]

    matched_skills = (matrix.loc[top_job].loc[lambda x: x>0].sort_values(ascending=False))

    print(f"\nSkills for {title_lookup.get(job_title, job_title)}")
    print(selected_skills.head(15).to_string())
    
    print(f"\nSkills for {title_lookup.get(top_job, top_job)}")
    print(matched_skills.head(15).to_string())


Enter job title: data scientist



Top 15 similar jobs:
                       Job_Title Cosine_Similarity
                  Data Scientist            1.0000
         Program/Project Manager            0.9565
       Machine Learning Engineer            0.9492
                 Product Manager            0.9475
              Research Scientist            0.9310
               Software Engineer            0.9184
           DBA/Database Engineer            0.9143
                    Data Analyst            0.9063
    Developer Relations/Advocacy            0.9044
                    Statistician            0.8906
                Business Analyst            0.8892
                   Data Engineer            0.6835
           Senior Data Scientist            0.6700
Data Analyst - Surface Equipment            0.5370
            (USA) Data Scientist            0.5196
             Senior Data Analyst            0.4966
    Senior Business Data Analyst            0.4919
                Data Analyst III            0.4727
        D